# 第五阶段：智能指针

## 实验 8：Owner 与 Borrower 分离

现实 SDK 不会让所有函数都接收智能指针。智能指针主要描述谁控制生命周期；普通业务函数通常只需要在调用期间使用对象。ownership 与 object access 是两个不同问题，应由不同的参数类型表达。

本阶段的最后一个实验关注：

- `unique_ptr` owner 如何向函数提供引用或 pointer 借用；
- 必须存在、可空、只读和可写借用如何通过类型区分；
- 为什么智能指针应集中在 factory、构造函数、成员和 ownership sink；
- borrower 何时失效，为什么同步借用不能被长期保存；
- `unique_ptr` 与 `shared_ptr` owner 如何复用同一个普通业务 API；
- C++ RAII owner 如何转换为 C ABI opaque handle 与 Kotlin/Native owner。

运行方式：重启 `xcpp23` kernel 后从上到下执行所有单元，并观察每个 owner 作用域结束时的析构日志。

In [ ]:
// 本步骤：引入智能指针、字符串、定宽整数、断言和移动所需的标准库。
#include <cassert>
#include <cstdint>
#include <iostream>
#include <memory>
#include <stdexcept>
#include <string>
#include <utility>

### 1. 准备由 owner 管理的业务对象

`Engine` 不知道自己由栈对象、`unique_ptr`、`shared_ptr` 还是某个上层成员拥有。它只实现业务行为；生命周期策略由外部 owner 决定。

In [ ]:
// 本步骤：定义与具体智能指针解耦的 Engine，并记录生命周期。
class Engine
{
public:
    Engine(int id, std::string mode)
        : id_(id), mode_(std::move(mode))
    {
        // 构造开始 Engine 生命周期，输出当前存活数量。
        ++live_count_;
        std::cout << "Engine " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~Engine() noexcept
    {
        // owner 结束生命周期时触发析构；borrower 不负责执行这一步。
        --live_count_;
        std::cout << "Engine " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    std::int32_t execute() const
    {
        // const 操作只读取 Engine，返回定宽状态值便于后续 C ABI 映射。
        std::cout << "execute Engine " << id_ << " in " << mode_ << " mode\n";
        return id_ * 10;
    }

    void set_mode(std::string mode)
    {
        // 可变操作更新业务状态，但不改变 Engine ownership。
        mode_ = std::move(mode);
    }

    const std::string &mode() const noexcept
    {
        // 返回内部字符串的只读借用，有效期不超过 Engine。
        return mode_;
    }

    static int live_count() noexcept
    {
        // 返回当前存活 Engine 数量，供实验断言使用。
        return live_count_;
    }

private:
    int id_;
    std::string mode_;
    inline static int live_count_ = 0;
};

### 2. 普通业务函数只接收所需借用

以下三种借用都不参与 ownership：

| 参数 | contract |
| --- | --- |
| `const Engine&` | 必须存在、同步只读借用 |
| `Engine&` | 必须存在、同步可写借用 |
| `const Engine*` | 可为空、同步只读借用 |

如果函数只关心 `Engine`，就不应让它依赖调用方恰好使用哪一种智能指针。

In [ ]:
// 本步骤：定义必须存在的只读借用接口。
std::int32_t run(const Engine &engine)
{
    // 在调用期间使用 Engine，不保存引用，也不取得 ownership。
    return engine.execute();
}

// 本步骤：定义必须存在的可写借用接口。
void configure(Engine &engine, std::string mode)
{
    // 修改现有对象状态；对象仍由原 owner 管理。
    engine.set_mode(std::move(mode));
}

// 本步骤：定义可空的只读借用接口。
std::int32_t maybe_run(const Engine *engine)
{
    // pointer 为空时没有可执行对象，因此返回约定的错误状态。
    if (engine == nullptr)
    {
        return -1;
    }

    // 非空 pointer 只在本次调用期间借用 Engine。
    return run(*engine);
}

In [ ]:
// 本步骤：由 unique_ptr 提供三种借用，并确认 owner 始终不变。
{
    const int before = Engine::live_count();
    auto owner = std::make_unique<Engine>(1, "safe");

    // 解引用 unique_ptr 得到 Engine&，按 const 参数只读借用。
    assert(run(*owner) == 10);
    assert(owner);

    // 同一个 Engine& 也可传给可写借用接口，不转移 ownership。
    configure(*owner, "fast");
    assert(owner->mode() == "fast");

    // get() 提供可空 pointer 借用；owner 仍负责最终析构。
    assert(maybe_run(owner.get()) == 10);
    assert(maybe_run(nullptr) == -1);
    assert(Engine::live_count() == before + 1);
}

离开代码块时 `owner` 析构并销毁 `Engine`。`run()`、`configure()` 和 `maybe_run()` 都没有改变生命周期。关系是：

```text
unique_ptr<Engine> owner
        │ owns
        ▼
      Engine
        ▲
        │ borrows during call
run(const Engine&)
```

### 3. 不要把智能指针传播到普通业务 API

不推荐仅为了调用 `execute()` 而声明：

```cpp
void run(const std::unique_ptr<Engine> &engine);
```

这个签名把业务函数耦合到 `unique_ptr`，却没有取得 ownership；它还被迫处理智能指针可能为空的问题。若参数必须存在，`const Engine&` 的 contract 更直接。

只有函数真正关心智能指针本身时才接收它，例如 factory 返回 owner、构造函数接管 owner、容器保存 owner，或 ownership sink 消费 owner。

### 4. 智能指针集中在 ownership boundary

`Runtime` 的成员负责拥有 `Engine`。构造函数按值接收 `unique_ptr`，明确接管 ownership；成员函数内部则把普通引用传给业务函数。

In [ ]:
// 本步骤：定义返回唯一 ownership 的 Engine factory。
std::unique_ptr<Engine> create_engine(int id, std::string mode)
{
    // 返回类型要求调用方接管并管理 Engine 生命周期。
    return std::make_unique<Engine>(id, std::move(mode));
}

// 本步骤：定义拥有 Engine 的 Runtime，将智能指针限制在 ownership boundary。
class Runtime
{
public:
    explicit Runtime(std::unique_ptr<Engine> engine)
        : engine_(std::move(engine))
    {
        // Runtime 要求取得非空 ownership，失败应在边界立即报告。
        if (!engine_)
        {
            throw std::invalid_argument("Runtime requires an Engine");
        }
    }

    std::int32_t execute() const
    {
        // 内部业务调用只传 Engine 引用，不传播 unique_ptr 类型。
        return run(*engine_);
    }

    const Engine &engine() const noexcept
    {
        // getter 返回只读借用，不能超过 Runtime 生命周期。
        return *engine_;
    }

private:
    std::unique_ptr<Engine> engine_;
};

In [ ]:
// 本步骤：追踪 ownership 从 factory 转移到 Runtime，再由 Runtime 提供借用。
{
    const int before = Engine::live_count();

    {
        // factory 返回 owner，pending 暂时负责 Engine 生命周期。
        auto pending = create_engine(2, "balanced");

        // Runtime 构造函数消费 ownership，pending 随后变空。
        Runtime runtime(std::move(pending));
        assert(!pending);
        assert(runtime.execute() == 20);

        // getter 的引用只是借用，Runtime 仍是唯一 owner。
        assert(runtime.engine().mode() == "balanced");
        assert(Engine::live_count() == before + 1);
    }

    // Runtime 析构带动成员 unique_ptr 析构，Engine 生命周期结束。
    assert(Engine::live_count() == before);
}

### 5. Borrower 的有效期不能超过 owner

引用和 raw pointer 不会自动追踪 owner。下面的返回值在函数结束时立即悬空：

```cpp
const Engine &bad_engine()
{
    auto owner = std::make_unique<Engine>(3, "temporary");
    return *owner; // owner 析构后，返回引用悬空
}
```

另一个常见错误是把 `owner.get()` 保存到异步任务或长期成员中，然后先销毁 owner。短期 borrower 只有在以下条件成立时才安全：

```text
borrow lifetime <= owner lifetime
```

如果任务必须独立延长生命周期，就应明确转移 `unique_ptr` 或共享 `shared_ptr`；如果任务只应在对象仍存在时执行，则保存 `weak_ptr` 并在使用前 `lock()`。

### 6. 同一业务 API 可以接受不同 owner 的借用

`run(const Engine&)` 不关心上游 ownership 策略，所以栈对象、`unique_ptr` 和 `shared_ptr` 都能调用它。这降低了耦合，也避免无意义的引用计数操作。

In [ ]:
// 本步骤：用三种 owner 调用同一个借用 API，验证业务函数与 ownership 解耦。
{
    Engine stack_owner(3, "stack");
    auto unique_owner = std::make_unique<Engine>(4, "unique");
    auto shared_owner = std::make_shared<Engine>(5, "shared");
    const long shared_count = shared_owner.use_count();

    // 三个表达式都产生同步的 const Engine& borrower。
    assert(run(stack_owner) == 30);
    assert(run(*unique_owner) == 40);
    assert(run(*shared_owner) == 50);

    // 只借用 shared object 不会增加 strong reference count。
    assert(shared_owner.use_count() == shared_count);
}

### 7. 从函数签名直接翻译 ownership contract

| 签名 | 应直接读出的语义 |
| --- | --- |
| `void inspect(const Engine &)` | 借用必须存在的只读 Engine |
| `void configure(Engine &)` | 借用必须存在的可写 Engine |
| `void maybe_run(const Engine *)` | 借用可能为空的只读 Engine |
| `unique_ptr<Engine> create()` | 返回唯一 ownership |
| `void consume(unique_ptr<Engine>)` | 取得并消费唯一 ownership |
| `void retain(shared_ptr<Engine>)` | 建立共享 ownership，并可能长期保留 |
| `void observe(weak_ptr<Engine>)` | 保存可检查但不保活的观察关系 |

智能指针出现在参数中时，它应该说明真正的 ownership 行为，而不是只充当访问对象的语法包装。

### 8. C ABI 使用 opaque handle 表达边界

`unique_ptr<Engine>` 适合 C++ 内部，但不能直接作为 C ABI 类型。跨边界时把 owner 封装在原生实现中，并只暴露不透明 handle：

```c
typedef struct sdk_engine_handle sdk_engine_handle;

sdk_engine_handle *sdk_engine_create(void);  // 返回 owning handle
int32_t sdk_engine_execute(
    const sdk_engine_handle *handle          // 同步借用 handle
);
void sdk_engine_destroy(
    sdk_engine_handle *handle                // 消费 ownership
);
```

下面在 C++ 中实现这一边界。`new/delete` 被限制在成对的 ABI lifecycle 函数里；C++ Core 的普通业务逻辑仍使用引用。

In [ ]:
// 本步骤：定义只在 C++ 实现中可见的 opaque handle 完整布局。
struct sdk_engine_handle
{
    // handle 直接拥有 Engine 值，销毁 handle 会自动析构 Engine。
    Engine engine{6, "abi"};
};

// 本步骤：实现 C ABI factory，成功时把 handle ownership 交给调用方。
extern "C" sdk_engine_handle *sdk_engine_create() noexcept
{
    try
    {
        // ABI 边界允许集中使用 new，并由 sdk_engine_destroy 成对释放。
        return new sdk_engine_handle{};
    }
    catch (...)
    {
        // C++ 异常不能跨 C ABI，失败转换为空 handle。
        return nullptr;
    }
}

// 本步骤：实现只同步借用 handle 的 C ABI 业务函数。
extern "C" std::int32_t sdk_engine_execute(const sdk_engine_handle *handle) noexcept
{
    // C pointer 可能为空，边界必须先验证再借用内部 Engine。
    if (handle == nullptr)
    {
        return -1;
    }

    // 本次调用只借用 handle 内的 Engine，不改变 handle ownership。
    return run(handle->engine);
}

// 本步骤：实现消费 owning handle 的 C ABI 销毁函数。
extern "C" void sdk_engine_destroy(sdk_engine_handle *handle) noexcept
{
    // delete nullptr 安全；非空时同时析构 handle 及其 Engine 成员。
    delete handle;
}

In [ ]:
// 本步骤：模拟 C 调用方，验证 create/borrow/destroy 的完整生命周期。
{
    const int before = Engine::live_count();

    // create 返回 owning raw handle；调用方必须记录销毁责任。
    sdk_engine_handle *handle = sdk_engine_create();
    assert(handle != nullptr);
    assert(Engine::live_count() == before + 1);

    // execute 只在调用期间借用 handle，调用后 handle 仍由调用方拥有。
    assert(sdk_engine_execute(handle) == 60);
    assert(sdk_engine_execute(nullptr) == -1);

    // destroy 消费 handle ownership；之后原地址失效，不能再次使用。
    sdk_engine_destroy(handle);
    handle = nullptr;
    assert(Engine::live_count() == before);
}

### 9. Kotlin/Native wrapper 重新建立 owner 语义

cinterop 看到的是 `CPointer<sdk_engine_handle>`，它不会自动知道该调用 `sdk_engine_destroy()`。Kotlin 包装层应重新建立明确的生命周期协议：

```text
Kotlin Engine owner
        │ owns
        ▼
CPointer<sdk_engine_handle>
        │ execute() 时同步借用
        │ close() 时消费 ownership
        ▼
C ABI create / execute / destroy
        │
        ▼
C++ Engine / RAII implementation
```

wrapper 需要防止 close 后继续调用、防止重复 destroy，并明确并发与回调是否允许超过一次方法调用。GC 管理 Kotlin 对象引用，不等于自动管理 C/C++ handle。

## 实验结论

智能指针应主要出现在 ownership boundary，而不是扩散到所有业务 API：

```text
create_engine() -> unique_ptr<Engine>
        │ ownership transfer
        ▼
Runtime::engine_
        │ const Engine& borrow
        ▼
run() / execute()
```

验收时应能直接回答：

1. owner 创建、保存并最终释放资源；borrower 只在约定时间内访问；
2. `T&` 表示必须存在的借用，`T*` 通常表示可能为空的借用；
3. `unique_ptr<T>` 表示唯一 ownership，`shared_ptr<T>` 表示共享 ownership；
4. `weak_ptr<T>` 表示不保活、使用前需要 `lock()` 的观察关系；
5. `get()` 不转移 ownership，`release()` 才会放弃 owner 的自动管理；
6. borrower 失效时间由 owner 生命周期决定；
7. C ABI 不能暴露 C++ 智能指针，应使用 handle 与成对 lifecycle 函数；
8. Kotlin/Native wrapper 必须重新实现 close/destroy 的 owner contract。

最终原则：**函数只需要使用对象时传对象；只有函数真正参与 ownership 时才传智能指针。**